In [ ]:
import pandas as pd
import os
import glob
from datetime import datetime

# 1. LOAD FILE LIST
# If you saved the CSV from the previous step:
# df = pd.read_csv("mea_file_locations.csv")

# OR, if you want to run it fresh right now:
root_search_path = "/pscratch/sd/m/mpatil1/MEA_Analysis/AnalyzedData/V1_PYTHON/CDKL5_T1"
files = glob.glob(os.path.join(root_search_path, "**", "raster_burst_plot_60s.svg"), recursive=True)
df = pd.DataFrame(files, columns=['full_path'])
ANCHOR = "V1_PYTHON"  # this is the folder that anchors the structure we want to parse

def parse_path_metadata(path):
    """
    Extracts metadata from the standard path structure:
    .../AnalyzedData/Genotype/Date/ChipID/Network/RunID/WellID/...
    """
    try:
        parts = path.split(os.sep)
        
        # We anchor everything relative to "AnalyzedData" to be safe
        if ANCHOR in parts:
            idx = parts.index(ANCHOR)
            
            # Extract based on your specific directory depth
            project = parts[idx + 1]  # e.g., CDKL5_T1
            date_str = parts[idx + 2]  # e.g., 240531
            chip_id  = parts[idx + 3]  # e.g., M07420
            # parts[idx + 4] is usually "Network"
            run_id   = parts[idx + 5]  # e.g., 000052
            well_id  = parts[idx + 6]  # e.g., well005
            
            return pd.Series([project, date_str, chip_id, run_id, well_id])
        else:
            return pd.Series([None, None, None, None, None])
            
    except IndexError:
        return pd.Series([None, None, None, None, None])

# 2. APPLY EXTRACTION
print(" dissecting path names...")
metadata_cols = ["Project", "Date", "Chip_ID", "RunID", "Well"]
df[metadata_cols] = df['full_path'].apply(parse_path_metadata)
#puth full_path to last column
df = df[[col for col in df.columns if col != 'full_path'] + ['full_path']]


# 4. REVIEW
print(f"Successfully dissected {len(df)} paths.")
print("\nSnapshot of extracted metadata:")
print(df.head())

# 5. SAVE
#df.to_csv("/pscratch/sd/m/mpatil1/MEA_Analysis/IPNAnalysis/workbooks/mea_metadata_index_rasters.csv", index=False)

In [ ]:
import pandas as pd

ref_df = pd.read_excel("/pscratch/sd/m/mpatil1/Data/CDKL5_T1/CDKL5_T1_C1_reff_new.xlsx")
print(ref_df.columns)

# --- 2. TRANSFORMATION LOGIC ---

# Step A: Split the strings into actual Python lists
ref_df['Wells_List'] = ref_df['Wells_Recorded'].astype(str).str.split(',')
ref_df['Source_List'] = ref_df['Neuron Source'].astype(str).str.split(',')

# Step B: Explode the lists into rows (keeps alignment)
exploded_df = ref_df.explode(['Wells_List', 'Source_List'])

# Step C: Cleanup
exploded_df['Well'] = exploded_df['Wells_List'].str.strip()
exploded_df['NeuronType'] = exploded_df['Source_List'].str.strip()

# Step D: FIX — convert 1-based wells to 0-based and format
exploded_df['Well'] = (
    'well'
    + (exploded_df['Well'].astype(int) - 1)
      .astype(str)
      .str.zfill(3)
)

# Filter to keep only the clean columns
clean_ref_df = exploded_df[['Date', 'ID', 'Run #', 'Well', 'NeuronType', 'Assay', 'DIV']]

# --- 3. VIEW RESULTS ---
print("Transformation complete.")
print(f"Original rows: {len(ref_df)}")
print(f"Exploded rows: {len(clean_ref_df)}")
print("\nFirst 10 rows of clean metadata:")
print(clean_ref_df.head(10))

# Save if needed
# clean_ref_df.to_csv("clean_metadata_map.csv", index=False)

In [ ]:
df.columns

In [ ]:
clean_ref_df.columns

In [ ]:
clean_ref_df['Date'] = (
    pd.to_datetime(clean_ref_df['Date'])
      .dt.strftime('%y%m%d')
)

In [ ]:
df.rename(columns={"RunID": "Run #","Chip_ID": "ID"}, inplace=True)

In [ ]:
df["Date"] = df["Date"].astype(str)

In [ ]:
df['Run #'] = df['Run #'].astype(str).str.zfill(6)
clean_ref_df['Run #'] = clean_ref_df['Run #'].astype(str).str.zfill(6)

In [ ]:
merged_df = pd.merge(
    df,
    clean_ref_df,
    how='left',
    on=['ID', 'Date', 'Run #', 'Well'],
    validate='one_to_one'
)

In [ ]:
#delete rows where DIV is NaN
merged_df = merged_df.dropna(subset=['DIV'])

In [ ]:
#split merged_df into two dataframes based on Assay type
network_assay_df = merged_df[merged_df['Assay'] == 'Network Today']
neuronal_assay_df = merged_df[merged_df['Assay'] == "Neuronal Units 9"]

In [ ]:
network_assay_df.head()

In [ ]:
merged_df['NeuronType'].unique()

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
from svglib.svglib import svg2rlg
from reportlab.graphics import renderPM
from tqdm import tqdm  # <--- Import tqdm

# --- 1. SETUP DATA ---

merged_df = network_assay_df.copy()

# Create Labels
merged_df['Sample_Label'] = merged_df['NeuronType'] + "\n" + merged_df['ID'] + "_" + merged_df['Well']

# --- 2. GRID SETUP ---
unique_divs = sorted(merged_df['DIV'].unique())
unique_samples = merged_df['Sample_Label'].unique()

n_rows = len(unique_divs)
n_cols = len(unique_samples)

print(f"Grid Layout: {n_rows} Rows (DIVs) x {n_cols} Cols (Samples)")
print(f"Total SVGs to render: {n_rows * n_cols}")

# --- 3. PLOT WITH PROGRESS BAR ---
fig, axs = plt.subplots(n_rows, n_cols, figsize=(3.0 * n_cols, 3.0 * n_rows), squeeze=False)

# Wrap the outer loop with tqdm for the status bar
for j, sample_label in enumerate(tqdm(unique_samples, desc="Processing Samples")):
    
    # Filter for this sample
    sample_data = merged_df[merged_df['Sample_Label'] == sample_label]
    axs[0][j].set_title(sample_label, fontsize=10, weight='bold', pad=15)
    
    for i, div in enumerate(unique_divs):
        ax = axs[i][j]
        row = sample_data[sample_data['DIV'] == div]
        
        if not row.empty:
            svg_path = row.iloc[0]['full_path']
            try:
                # Render SVG to PIL Image
                drawing = svg2rlg(svg_path)
                pil_img = renderPM.drawToPIL(drawing)
                
                ax.imshow(pil_img)
                ax.axis('off')
            except Exception as e:
                # Simple error handling
                ax.text(0.5, 0.5, "Error", ha='center', va='center', color='red', fontsize=8)
                ax.axis('off')
        else:
            # Placeholder for missing data
            ax.set_facecolor("#f5f5f5")
            ax.text(0.5, 0.5, "No Rec", ha='center', va='center', color='gray', fontsize=8, alpha=0.5)
            ax.set_xticks([]) 
            ax.set_yticks([])
            for spine in ax.spines.values(): spine.set_visible(False)

# --- 4. FINALIZE ---
print("Adding labels and saving...")
for i, div in enumerate(unique_divs):
    axs[i][0].annotate(
        f"DIV {div}",
        xy=(0, 0.5), xycoords='axes fraction',
        xytext=(-0.2, 0.5), textcoords='axes fraction',
        ha='right', va='center', fontsize=12, fontweight='bold'
    )

plt.subplots_adjust(wspace=0.05, hspace=0.05, left=0.1, right=0.98, top=0.92, bottom=0.02)
output_pdf = "/pscratch/sd/m/mpatil1/MEA_Analysis/AnalyzedData/V1_PYTHON/CDKL5_T1/Raster_Overview_Grid_NeuronalScan.pdf"
plt.savefig(output_pdf, dpi=150)
print(f"✅ Done! Saved to: {output_pdf}")

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
from svglib.svglib import svg2rlg
from reportlab.graphics import renderPM
from tqdm import tqdm  # <--- Import tqdm

# --- 1. SETUP DATA ---

merged_df = neuronal_assay_df.copy()

# Create Labels
merged_df['Sample_Label'] = merged_df['NeuronType'] + "\n" + merged_df['ID'] + "_" + merged_df['Well']

# --- 2. GRID SETUP ---
unique_divs = sorted(merged_df['DIV'].unique())
unique_samples = merged_df['Sample_Label'].unique()

n_rows = len(unique_divs)
n_cols = len(unique_samples)

print(f"Grid Layout: {n_rows} Rows (DIVs) x {n_cols} Cols (Samples)")
print(f"Total SVGs to render: {n_rows * n_cols}")

# --- 3. PLOT WITH PROGRESS BAR ---
fig, axs = plt.subplots(n_rows, n_cols, figsize=(3.0 * n_cols, 3.0 * n_rows), squeeze=False)

# Wrap the outer loop with tqdm for the status bar
for j, sample_label in enumerate(tqdm(unique_samples, desc="Processing Samples")):
    
    # Filter for this sample
    sample_data = merged_df[merged_df['Sample_Label'] == sample_label]
    axs[0][j].set_title(sample_label, fontsize=10, weight='bold', pad=15)
    
    for i, div in enumerate(unique_divs):
        ax = axs[i][j]
        row = sample_data[sample_data['DIV'] == div]
        
        if not row.empty:
            svg_path = row.iloc[0]['full_path']
            try:
                # Render SVG to PIL Image
                drawing = svg2rlg(svg_path)
                pil_img = renderPM.drawToPIL(drawing)
                
                ax.imshow(pil_img)
                ax.axis('off')
            except Exception as e:
                # Simple error handling
                ax.text(0.5, 0.5, "Error", ha='center', va='center', color='red', fontsize=8)
                ax.axis('off')
        else:
            # Placeholder for missing data
            ax.set_facecolor("#f5f5f5")
            ax.text(0.5, 0.5, "No Rec", ha='center', va='center', color='gray', fontsize=8, alpha=0.5)
            ax.set_xticks([]) 
            ax.set_yticks([])
            for spine in ax.spines.values(): spine.set_visible(False)

# --- 4. FINALIZE ---
print("Adding labels and saving...")
for i, div in enumerate(unique_divs):
    axs[i][0].annotate(
        f"DIV {div}",
        xy=(0, 0.5), xycoords='axes fraction',
        xytext=(-0.2, 0.5), textcoords='axes fraction',
        ha='right', va='center', fontsize=12, fontweight='bold'
    )

plt.subplots_adjust(wspace=0.05, hspace=0.05, left=0.1, right=0.98, top=0.92, bottom=0.02)
output_pdf = "/pscratch/sd/m/mpatil1/MEA_Analysis/AnalyzedData/V1_PYTHON/CDKL5_T1//Raster_Overview_Grid_NeuronalScan.pdf"
plt.savefig(output_pdf, dpi=150)
print(f"✅ Done! Saved to: {output_pdf}")